# 🔥 PyTorch Key Capabilities — Practice Notebook

A hands-on, end-to-end tour of PyTorch:

1. Tensors and operations
2. Autograd
3. Loss functions
4. **Full training loop from scratch** (linear regression)
5. `nn.Module` and key layers
6. **Full professional training loop** on MNIST (simple MLP)
7. Device handling, eval mode, save/load
8. Practice exercises

The mantra:

$$
\text{forward} \;\rightarrow\; \text{loss} \;\rightarrow\; \text{backward} \;\rightarrow\; \text{step}
$$

Update rule:

$$
w \leftarrow w - lr \cdot \frac{dL}{dw}
$$

## 🛠️ 0. Setup

- Import PyTorch.
- Set a seed for reproducibility.
- Pick a device once and reuse it everywhere.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", device)

PyTorch: 2.11.0+cu130
Device : cpu


/home/erradi/.venvs/ml-env/lib/python3.14/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 🧱 1. Tensors

Tensors are n-dimensional numerical arrays with three key fields:

- `shape` — dimensions
- `dtype` — numeric type
- `device` — where it lives

Three common creation patterns: from data, from a shape, and `_like` an existing tensor.

In [2]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
zeros_tensor = torch.zeros(2, 3)
random_tensor = torch.randn(2, 3)
zeros_like_a_tensor  = torch.zeros_like(a)

print("a shape:", a.shape, "dtype:", a.dtype, "device:", a.device)
print("zeros tensor:\n", zeros_tensor)
print("random tensor:\n", random_tensor)
print("zeros like(a):\n", zeros_like_a_tensor)

a shape: torch.Size([2, 2]) dtype: torch.float32 device: cpu
zeros tensor:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
random tensor:
 tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986]])
zeros like(a):
 tensor([[0., 0.],
        [0., 0.]])


## 🧮 2. Operations: `*` vs `@`

- `a * b` — element-wise (same shape).
- `a @ b` — matrix multiplication (inner dimensions must match).

A linear layer is just:

$$
\hat{y} = X w + b
$$



\begin{aligned}
\text{where:} \quad
X &\in \mathbb{R}^{m \times n} \;\;&&\text{input dataset matrix}, \\
m &\;\;=\;\; &&\text{number of samples}, \\
n &\;\;=\;\; &&\text{number of input features}, \\
\mathbf{w} &\in \mathbb{R}^{n} \;\;&&\text{\(n\)-dimensional weight vector}, \\
b &\in \mathbb{R} \;\;&&\text{bias term (scalar)}, \\
\hat{\mathbf{y}} &\in \mathbb{R}^{m} \;\;&&\text{\(m\)-dimensional output vector}.
\end{aligned}

"Output Position","Vectors Used","Dot Product Calculation","Result"
"Top-Left","Row 1 of A [1., 2.]Col 1 of B [10., 30.]","(1.0 * 10.0) + (2.0 * 30.0)  = 10.0 + 60.0","70.0"
"Top-Right","Row 1 of A [1., 2.]Col 2 of B [20., 40.]","(1.0 * 20.0) + (2.0 * 40.0)  = 20.0 + 80.0","100.0"
"Bottom-Left","Row 2 of A [3., 4.]Col 1 of B [10., 30.]","(3.0 * 10.0) + (4.0 * 30.0)  = 30.0 + 120.0","150.0"
"Bottom-Right","Row 2 of A [3., 4.]Col 2 of B [20., 40.]","(3.0 * 20.0) + (4.0 * 40.0)  = 60.0 + 160.0","220.0"

In [3]:
A = torch.tensor([[1., 2.], 
                  [3., 4.]])
B = torch.tensor([[10., 20.], 
                  [30., 40.]])

# Element-wise (*): Multiplies elements at matching indices. Shapes must match.
print("Element-wise Multiplication A * B:\n", A * B)

# Matrix Multiplication (@): Computes the dot product (sum of products) between 
# the rows of matrix A and the columns of matrix B. 
# The number of columns in A must equal the number of rows in B (n, m) @ (m, p) -> (n, p).
print("Matrix Multiplication (matmul) A @ B:\n", A @ B)

# Linear Layer: y_hat = X @ w + b
X = torch.randn(10, 4)  # Input dataset: 10 samples, 4 features
w = torch.randn(4)      # 4 weights
b = torch.randn(1)      # 1 bias
y_hat = X @ w + b

print("X:\n", X)
print("w:\n", w)
print("b:\n", b)
print("y_hat shape:", y_hat.shape)
print("y_hat:\n", y_hat)

Element-wise Multiplication A * B:
 tensor([[ 10.,  40.],
        [ 90., 160.]])
Matrix Multiplication (matmul) A @ B:
 tensor([[ 70., 100.],
        [150., 220.]])
X:
 tensor([[-0.0885,  0.5239, -0.6659,  0.8504],
        [-1.3527, -1.6959,  0.5667,  0.7935],
        [-0.1932, -0.3090,  0.5026, -0.8594],
        [ 0.7502, -0.5855, -0.1734,  0.1835],
        [ 0.1665,  0.8744, -0.1435, -0.1116],
        [-0.6136,  0.0316, -0.4927,  0.2484],
        [-0.2303, -0.3918,  0.5433, -0.3952],
        [ 0.2055, -0.4503, -0.5731, -0.5554],
        [-1.5312, -1.2341,  1.8197, -0.5515],
        [-1.3253,  0.1886, -0.0691, -0.4949]])
w:
 tensor([-2.2188,  0.2590, -1.0297, -0.5008])
b:
 tensor([0.2734])
y_hat shape: torch.Size([10])
y_hat:
 tensor([ 0.8653,  1.8546,  0.5349, -1.4561,  0.3341,  2.0259,  0.3214,  0.5689,
         1.7535,  3.5818])


## 📐 3. Reductions and `dim`

The `dim` you pass is the dimension that **disappears** from the output.

- `dim=0` → collapse rows → one value per column.
- `dim=1` → collapse columns → one value per row.
- `dim=-1` → collapse the last axis (very common for batches and features).

In [4]:
# Scores: 2 students, 3 assignments
scores = torch.tensor([
    [10., 20., 30.],
    [ 5., 10., 15.],
])

# Scores shape: 2 students, 3 assignments
print("scores shape:", scores.shape)

# Scores dimension: 2D . 0-indexed 0 refers to rows, 1 refers to columns
print("scores dim  :", scores.dim())

# Sum of scores per assignment (0-indexed dimension)
print("sum(dim=0) per assignment:", scores.sum(dim=0))

# Sum of scores per student (1-indexed dimension)
print("sum(dim=1) per student   :", scores.sum(dim=1))

# Sum of scores per assignment (last dimension)
print("sum(dim=-1) per assignment:", scores.sum(dim=-1))

# Mean score per assignment (last dimension)
print("mean(dim=-1) per assignment:", scores.mean(dim=-1))

scores shape: torch.Size([2, 3])
scores dim  : 2
sum(dim=0) per assignment: tensor([15., 30., 45.])
sum(dim=1) per student   : tensor([60., 30.])
sum(dim=-1) per assignment: tensor([60., 30.])
mean(dim=-1) per assignment: tensor([20., 10.])


## ⚡ 4. Autograd

PyTorch builds a dynamic computation graph during the forward pass.
`loss.backward()` computes derivatives via the chain rule:

$$
\frac{dL}{dw} = \frac{dL}{d\hat{y}} \cdot \frac{d\hat{y}}{dw}
$$

For $\hat{y} = w x + b$ with squared error $L = (\hat{y} - y)^2$ we expect:

$$
\frac{dL}{dw} = 2(\hat{y}-y)\,x, \qquad \frac{dL}{db} = 2(\hat{y}-y)
$$

In [5]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
x = torch.tensor(3.0)
y = torch.tensor(7.0)

y_hat = w * x + b
loss = (y_hat - y) ** 2
loss.backward()

print("y_hat:", y_hat.item(), "loss:", loss.item())
print("w.grad:", w.grad.item(), "(expected:", (2 * (y_hat.item() - y.item()) * x.item()), ")")
print("b.grad:", b.grad.item(), "(expected:", (2 * (y_hat.item() - y.item())), ")")

y_hat: 6.0 loss: 1.0
w.grad: -6.0 (expected: -6.0 )
b.grad: -2.0 (expected: -2.0 )


## 🎯 5. Loss Functions

- Regression: `nn.MSELoss` — $L = \frac{1}{N}\sum (\hat{y}_i - y_i)^2$
- Classification: `nn.CrossEntropyLoss` — expects **raw logits**, not softmax probabilities.

$$
L_{CE} = -\log \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}
$$

In [6]:
mse = nn.MSELoss()
ce  = nn.CrossEntropyLoss()

print("MSE:", mse(torch.randn(8, 1), torch.randn(8, 1)).item())

logits = torch.randn(8, 10)
labels = torch.randint(0, 10, (8,))
print("CE :", ce(logits, labels).item())

MSE: 2.3685123920440674
CE : 2.2411413192749023


## 🛠️ 6. Full Training Loop — From Scratch

We learn $y = 2x + 1$ from noisy data using **only** raw tensors and autograd.

The five steps:

1. Forward: `y_hat = X @ w + b`
2. Loss: `((y_hat - y) ** 2).mean()`
3. Backward: `loss.backward()`
4. Update: `w -= lr * w.grad`
5. Reset: `w.grad.zero_()`

In [7]:
torch.manual_seed(0)

N = 100
X = torch.randn(N, 1)
y = 2.0 * X + 1.0 + 0.1 * torch.randn(N, 1)

w = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.05
epochs = 200

for epoch in range(epochs):
    y_hat = X @ w + b
    loss  = ((y_hat - y) ** 2).mean()

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()
    b.grad.zero_()

    if epoch % 40 == 0:
        print(f"epoch={epoch:03d} loss={loss.item():.4f} "
              f"w={w.item():.3f} b={b.item():.3f}")

print(f"\nlearned: w={w.item():.3f}, b={b.item():.3f}  (true: 2.000, 1.000)")

epoch=000 loss=11.2178 w=-0.761 b=0.113
epoch=040 loss=0.0097 w=1.957 b=1.009
epoch=080 loss=0.0084 w=1.988 b=1.016
epoch=120 loss=0.0084 w=1.988 b=1.016
epoch=160 loss=0.0084 w=1.988 b=1.016

learned: w=1.988, b=1.016  (true: 2.000, 1.000)


## 🏗️ 7. `nn.Module` and Optimizer

The same regression — now with the professional pattern.

- `nn.Linear` holds $w$ and $b$.
- `optim.Adam` replaces the manual update.
- `optimizer.zero_grad()` replaces manual `.grad.zero_()`.

In [8]:
class LinearRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(1, 1)

    def forward(self, x):
        return self.layer(x)

model = LinearRegressor()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

for epoch in range(200):
    y_hat = model(X)
    loss  = criterion(y_hat, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

w_learned = model.layer.weight.item()
b_learned = model.layer.bias.item()
print(f"learned: w={w_learned:.3f}, b={b_learned:.3f}  (true: 2.000, 1.000)")

learned: w=1.988, b=1.016  (true: 2.000, 1.000)


## 🖼️ 8. MNIST Classifier — Simple MLP

A realistic example using a small MLP.

Architecture:

$$
\text{image (1×28×28)} \rightarrow \text{Flatten (784)} \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Dropout} \rightarrow \text{Linear} \rightarrow \text{logits (10)}
$$

In [9]:
class MNISTClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = MNISTClassifier().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {n_params:,}")

MNISTClassifier(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
trainable parameters: 101,770


## 📦 9. Data Pipeline

- `torchvision.datasets.MNIST` provides the dataset (downloaded once).
- `transforms` convert images to tensors and normalize them.
- `DataLoader` batches and shuffles examples.

> First run downloads MNIST (~10 MB) into `./data`.

In [10]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(root="data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

X_batch, y_batch = next(iter(train_loader))
print("X_batch:", X_batch.shape, X_batch.dtype)
print("y_batch:", y_batch.shape, y_batch.dtype)

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.46MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 156kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.02MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.7MB/s]

X_batch: torch.Size([128, 1, 28, 28]) torch.float32
y_batch: torch.Size([128]) torch.int64


## 🔁 10. Full Professional Training Loop

We split into `train_one_epoch` and `evaluate`:

- `model.train()` for training, `model.eval()` for evaluation.
- `torch.inference_mode()` disables gradient tracking during evaluation.
- We track both loss and accuracy.

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        logits = model(X)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total   += X.size(0)
    return total_loss / total, correct / total

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)

        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total   += X.size(0)
    return total_loss / total, correct / total

EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    print(f"epoch={epoch} "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} "
          f"test_loss={te_loss:.4f} test_acc={te_acc:.4f}")

epoch=1 train_loss=0.3476 train_acc=0.8984 test_loss=0.1607 test_acc=0.9508
epoch=2 train_loss=0.1702 train_acc=0.9494 test_loss=0.1139 test_acc=0.9662
epoch=3 train_loss=0.1301 train_acc=0.9609 test_loss=0.0938 test_acc=0.9729


## 🔮 11. Inference on a Single Batch

- Switch to eval mode.
- Use `torch.inference_mode()` for speed.
- Use `argmax(dim=1)` to turn logits into class predictions.

In [12]:
model.eval()
X_batch, y_batch = next(iter(test_loader))
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

with torch.inference_mode():
    logits = model(X_batch)
    probs  = logits.softmax(dim=1)
    preds  = logits.argmax(dim=1)

print("first 10 predictions:", preds[:10].tolist())
print("first 10 truths     :", y_batch[:10].tolist())
print("batch accuracy      :", (preds == y_batch).float().mean().item())

first 10 predictions: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
first 10 truths     : [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
batch accuracy      : 0.974609375


## 💾 12. Save and Load

Save the `state_dict` (parameter values), not the whole Python object.

- Reconstruct the model class.
- Load weights with `load_state_dict`.
- Use `weights_only=True` for safer deserialization.

In [13]:
torch.save(model.state_dict(), "mnist_mlp.pt")

reloaded = MNISTClassifier().to(device)
reloaded.load_state_dict(torch.load("mnist_mlp.pt", map_location=device, weights_only=True))
reloaded.eval()

with torch.inference_mode():
    same = (reloaded(X_batch).argmax(dim=1) == preds).all().item()
print("reloaded matches original predictions:", same)

reloaded matches original predictions: True


## ✅ Recap

You implemented:

1. Tensor creation, ops, and reductions.
2. Autograd and gradient checking.
3. A **full from-scratch** training loop (linear regression).
4. A **full professional** training loop (MNIST MLP).
5. Inference, saving, and loading.

The same five-step loop scales from this notebook to large models:

$$
\boxed{\;\text{forward} \;\rightarrow\; \text{loss} \;\rightarrow\; \text{backward} \;\rightarrow\; \text{step}\;}
$$

$$
w \leftarrow w - lr \cdot \frac{dL}{dw}
$$